# DATA 266 Lab 2 - Part 1\n## Standalone Colab notebook\nRun all cells from top to bottom. The notebook clones the repository, installs dependencies, runs a smoke test, generates the submission, and validates it.

In [ ]:
from pathlib import Path\nimport subprocess, sys, platform, random\nREPO_URL = 'https://github.com/denishatank12/Lab-2.git'\ncandidates = [Path.cwd(), *Path.cwd().parents]\nREPO_ROOT = next((p for p in candidates if (p / 'part1_rag').is_dir()), Path('/content/Lab-2'))\nif not (REPO_ROOT / 'part1_rag').is_dir():\n    subprocess.run(['git','clone','--depth','1',REPO_URL,str(REPO_ROOT)],check=True)\nPART1 = REPO_ROOT / 'part1_rag'\nDATA = PART1 / 'data'\nOUTPUTS = PART1 / 'outputs'\nOUTPUTS.mkdir(parents=True, exist_ok=True)\nsys.path.insert(0, str(PART1 / 'src'))\nprint('repository:', REPO_ROOT)\nprint('part1 data:', DATA)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','pypdf>=5.0','numpy>=1.26','sentence-transformers>=3.0'],check=True)\nprint('dependencies ready')

In [ ]:
import numpy as np\nSEED=266\nrandom.seed(SEED); np.random.seed(SEED)\nprint('python:', platform.python_version())\ntry:\n    import torch\n    print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())\n    if torch.cuda.is_available(): print('gpu:', torch.cuda.get_device_name(0))\nexcept Exception as exc: print('torch diagnostic unavailable:', exc)

In [ ]:
smoke=subprocess.run([sys.executable,str(PART1/'src'/'smoke_test.py')],text=True,capture_output=True)\nprint(smoke.stdout)\nif smoke.returncode != 0:\n    print(smoke.stderr)\n    raise RuntimeError('Smoke test failed; full run was stopped.')\nprint('SMOKE TEST PASSED')

In [ ]:
from rag_pipeline import build_index\nindex=build_index(DATA)\nprint('indexed passages:',len(index.passages))\nfor hit in index.search('What happened to unemployment after the crisis?',k=3):\n    p=hit['passage']; print(round(hit['score'],4),p['doc_id'],'page',p['page'])

In [ ]:
submission=OUTPUTS/'submission.csv'\nsubprocess.run([sys.executable,str(PART1/'src'/'generate_submission.py'),'--data-dir',str(DATA),'--output',str(submission)],check=True)\nsubprocess.run([sys.executable,str(DATA/'validate_submission.py'),str(submission),str(DATA/'Questions.csv')],check=True)\nprint('validated submission:',submission)\nprint('rows:',len(submission.read_text(encoding='utf-8').splitlines())-1)